<a href="https://colab.research.google.com/github/Lornatt88/maai902-nlp-solution/blob/main/notebooks/02_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q datasets transformers

from datasets import load_dataset
ds = load_dataset("coastalcph/lex_glue", "unfair_tos")

for split in ["train", "validation", "test"]:
    texts = ds[split]["text"]
    empty = sum(1 for t in texts if not t.strip())
    dupes = len(texts) - len(set(texts))
    print(f"{split}: {empty} empty, {dupes} duplicates")

README.md:   0%|          | 0.00/34.1k [00:00<?, ?B/s]

unfair_tos/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  501kB            

unfair_tos/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

unfair_tos/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  147kB            

unfair_tos/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

unfair_tos/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/5532 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1607 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2275 [00:00<?, ? examples/s]

train: 0 empty, 105 duplicates
validation: 0 empty, 14 duplicates
test: 0 empty, 7 duplicates


In [2]:
from collections import defaultdict, Counter

# 1. Most common duplicates in train
counts = Counter(ds["train"]["text"])
for text, n in counts.most_common(5):
    print(n, "x |", text[:80])

# 2. Same sentence, different labels?
labels_seen = defaultdict(set)
for t, l in zip(ds["train"]["text"], ds["train"]["labels"]):
    labels_seen[t].add(tuple(l))
conflicts = sum(1 for v in labels_seen.values() if len(v) > 1)
print("\nConflicting labels in train:", conflicts)

# 3. Test sentences that also appear in train
train_set = set(ds["train"]["text"])
overlap = sum(1 for t in ds["test"]["text"] if t in train_set)
print("Test sentences also in train:", overlap)

4 x | the copyright act provides that : 

3 x | please check the latest rates before you make your call . 

3 x | no refunds will be given for any charges more than 90 days old . 

3 x | you may opt out of this agreement to arbitrate . 

3 x | arbitration is more informal than a lawsuit in court . 


Conflicting labels in train: 1
Test sentences also in train: 33


As shown, there is some duplicates. But they are just common legal phrases that the companies reuse.

Also, Only 1 sentence in train has conflicting labels, which is too small to matter.

33 test sentences (about 2%) also appear in the training set. I kept the data unchanged so my results can be compared with the LexGLUE paper and I will check my results on the unseen test sentences later.

In [3]:
names = ds["train"].features["labels"].feature.names

for i, name in enumerate(names):
    print(f"\n=== {name} ===")
    examples = [t for t, l in zip(ds["train"]["text"], ds["train"]["labels"]) if i in l]
    for t in examples[:2]:
        print("-", t[:200])


=== Limitation of liability ===
- though tinder strives to encourage a respectful user experience through features like the double opt-in that only allows users to communicate if they have both indicated interest in one another , it i
- although tinder reserves the right to review and remove content that violates this agreement , such content is the sole responsibility of the user who posts it , and tinder can not guarantee that all 

=== Unilateral termination ===
- this means we may add new product features or enhancements from time to time as well as remove some features , and if these actions do not materially affect your rights or obligations , we may not pro
- tinder may terminate your account at any time without notice if it believes that you have violated this agreement . 


=== Unilateral change ===
- we may make changes to this agreement and to the services from time to time . 

- the most recent version of this agreement will be posted on the services under settings and als

Some of the sentences can belong to more than one label. Also the unfair part is often later on in the sentences like the middle of it or the end, therefore the model will need to read the whole sentence and not just look for a keyword.

In [4]:
from transformers import AutoTokenizer
import numpy as np

tok = AutoTokenizer.from_pretrained("bert-base-uncased")
lengths = [len(tok(t)["input_ids"]) for t in ds["train"]["text"]]

print("Average:", round(np.mean(lengths)))
print("95% of sentences are under:", int(np.percentile(lengths, 95)))
print("Longest:", max(lengths))
for limit in [64, 128, 256]:
    over = sum(1 for n in lengths if n > limit)
    print(f"Longer than {limit}: {over}")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Average: 38
95% of sentences are under: 85
Longest: 477
Longer than 64: 611
Longer than 128: 70
Longer than 256: 5


The average sentence is 38 tokens and 95% are under 85 tokens. Only 70 sentences (about 1.3%) are longer than 128 tokens so I chose a maximum length of 128. This fits almost all sentences while keeping training fast.

In [5]:
names = ds["train"].features["labels"].feature.names
num_labels = len(names)

def to_checklist(example):
    boxes = [0.0] * num_labels        # 8 empty boxes
    for label in example["labels"]:
        boxes[label] = 1.0            # tick each box that applies
    example["checklist"] = boxes
    return example

ds = ds.map(to_checklist)

# Check a few sentences: old label -> new checklist
for i in [0, 8, 10]:
    print(ds["train"][i]["labels"], "->", ds["train"][i]["checklist"])

# Check the total: should be 702 ticked boxes in train
total = sum(sum(row) for row in ds["train"]["checklist"])
print("Total ticked boxes in train:", int(total))

Map:   0%|          | 0/5532 [00:00<?, ? examples/s]

Map:   0%|          | 0/1607 [00:00<?, ? examples/s]

Map:   0%|          | 0/2275 [00:00<?, ? examples/s]

[] -> [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
[4] -> [0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0]
[2] -> [0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0]
Total ticked boxes in train: 702


I converted each sentence's labels into a checklist of 8 boxes, one per unfair type where 1 means the type applies and 0 means it doesn't.
Fair sentences have all zeros. This lets the model answer 8 separate yes/no questions per sentence.
The total number of ticked boxes (702) matches the original label count so the conversion is correct.

In [6]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize(batch):
    return tok(batch["text"], truncation=True, padding="max_length", max_length=128)

ds = ds.map(tokenize, batched=True)

# Look at one sentence after translation
ex = ds["train"][8]
print(ex["text"][:100])
print("\nTokens:", tok.convert_ids_to_tokens(ex["input_ids"])[:15])
print("IDs:", ex["input_ids"][:15])
print("Attention mask:", ex["attention_mask"][:15])
print("\nBox size:", len(ex["input_ids"]))
print("Real tokens:", sum(ex["attention_mask"]))

Map:   0%|          | 0/5532 [00:00<?, ? examples/s]

Map:   0%|          | 0/1607 [00:00<?, ? examples/s]

Map:   0%|          | 0/2275 [00:00<?, ? examples/s]

by creating a tinder account or by using the tinder imessage app ( `` tinder stacks '' ) , whether t

Tokens: ['[CLS]', 'by', 'creating', 'a', 'tin', '##der', 'account', 'or', 'by', 'using', 'the', 'tin', '##der', 'im', '##ess']
IDs: [101, 2011, 4526, 1037, 9543, 4063, 4070, 2030, 2011, 2478, 1996, 9543, 4063, 10047, 7971]
Attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]

Box size: 128
Real tokens: 120


I tokenized the text with the bert-base-uncased tokenizer. Each sentence starts with a [CLS] token and is padded or cut to 128 tokens. Words that are not in BERT's dictionary, like 'tinder' are split into smaller pieces.
An attention mask tells the model which tokens are real and which are padding.